# TP : Couches d'un datalake

Ce TP propose de reproduire localement l'organisation de données en couche au sein d'un datalake.

Rappel du modèle en couches présenté en cours (architecture "_medallion_"):

```
    Données entrantes => [Landing Zone] => [🥉 Bronze] => [🥈 Silver] => [🥇 Gold]
```

avec :
- _Landing Zone_ = espace de réception
- _Bronze_ = données brutes, non modifiées
- _Silver_ = données avec des modifications techniques de base
- _Gold_ = données raffinées avec des règles métier

Les couches sont simulées par des répertoires. On fournit la structure de répertoires suivante :

```
data
 ├─ landing
 │   ├─ diameters.zip
 │   └─ referentials.zip
 ├─ bronze
 │   ├─ diameters
 │   └─ referentials
 ├─ silver
 │   ├─ diameters
 │   └─ referentials
 └─ gold
```

Les fichiers ZIP dans `landing` sont donc les données déjà posées dans la _landing zone_.

Les répertoires `bronze` et `silver` contiennent chacun 2 sous-répertoires vides : `diameters` et `referentials`, un par source de données. On veillera dans la suite à créer les fichiers dans le bon sous-répertoire, selon la nature des données manipulées (référentiels ou diamètres). Il ne doit pas y avoir de fichiers directement dans `bronze` ou `silver`.

Le répertoire `gold` est vide au départ. Il va fusionner les sources donc pas besoin de sous-répertoire.

In [1]:
# Réglez la variable `DATA_DIR` avec l'emplacement où vous avez dézippé les données fournies :
# chemin du répertoire `data`
DATA_DIR = 'data'

In [2]:
# On prépare les chemins vers les répertoires de couches

import os

LANDING_DIR = os.path.join(DATA_DIR, 'landing')
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')
SILVER_DIR = os.path.join(DATA_DIR, 'silver')
GOLD_DIR = os.path.join(DATA_DIR, 'gold')

# Exemples d'utilisation :
# Chemin vers un fichier de réferentiel dans la couche "Silver" : os.path.join(SILVER_DIR, 'referentials', 'varieties.parquet')
# Chemin vers un fichier dans la couche "Gold" : os.path.join(GOLD_DIR, 'gold_ok.parquet')

In [3]:
import pandas as pd

## Passage vers la couche _Bronze_

On se contente de dézipper les fichiers sources vers le répertoire `bronze`.

In [4]:
import os
from zipfile import ZipFile

Voici le squelette de code pour les diamètres (voir les exemples ci-dessus pour la gestion des chemins) :

In [5]:
with ZipFile(os.path.join(LANDING_DIR, 'diameters.zip')) as landing_diameters:
    landing_diameters.extractall(os.path.join(BRONZE_DIR, 'diameters'))

Faire de même pour les référentiels :

In [6]:
with ZipFile(os.path.join(LANDING_DIR, 'referentials.zip')) as landing_referentials:
    landing_referentials.extractall(os.path.join(BRONZE_DIR, 'referentials'))

Vérifier dans le répertoire `bronze` que les données sont bien présentes.

## Passage vers la couche _Silver_

### Référentiels

Pour les référentiels, on fait confiance à la source et on se contentera de sauver les données à l'identique au format Parquet.

Charger les données de la couche _Bronze_ en mémoire avec Pandas, et les sauver sans modification au format Parquet (un fichier, pas de partitionnement, ne pas écrire l'index Pandas) dans le répertoire `silver/referentials`.

Vérifier que les données sont correctement chargées (attention au séparateur de champs CSV).

In [7]:
bronze_varieties = pd.read_csv(os.path.join(BRONZE_DIR, 'referentials', 'varieties.csv'), sep=',')

In [8]:
# "index=False" évite de sauver l'index Pandas (n° de ligne du dataframe)
bronze_varieties.to_parquet(os.path.join(SILVER_DIR, 'referentials', 'varieties.parquet'), index=False)

### Pour les diamètres

La qualité des données est variable et il va falloir les remettre d'aplomb.

Charger **toutes** les données de la couche _Bronze_ dans un seul gros dataframe, et l'explorer pour avoir une idée de la situation.

In [9]:
# Une manière de faire, avec "glob" qui renvoie une liste de fichiers
# On peut aussi tirer parti du fait qu'il y a 4 fichiers nommés diameters_1.csv, etc. et les charger explicitement

import glob

bronze_diameters_parts = [
    pd.read_csv(file)
    for file in glob.glob(os.path.join(BRONZE_DIR, 'diameters', '*.csv'))
]
bronze_diameters = pd.concat(bronze_diameters_parts)

In [10]:
bronze_diameters

,timestamp,variety_id,diameter,detection_status
0,2026-01-10 14:52:03,12,10.305,Ok
1,2026-01-10 14:53:00,7,8.995,yes
2,2026-01-10 14:54:05,12,10.099,Fail
3,2026-01-10 14:55:16,12,9.115,ERror
4,2026-01-10 14:56:12,16,9.875,FAILURE
...,...,...,...,...
35,2026-01-10 16:47:12,6,10.804,OK
36,2026-01-10 16:48:16,1,8.363,ERror
37,2026-01-10 16:49:15,2,10.880,Fail
38,2026-01-10 16:50:07,12,10.681,GOod


In [11]:
# Types des colonnes. On voit que `timestamp` n'est pas décodé en datetime
bronze_diameters.dtypes

timestamp            object
variety_id            int64
diameter            float64
detection_status     object
dtype: object

In [12]:
# Examen rapide pour voir s'il n'y a pas des valeurs aberrantes (si)
bronze_diameters.diameter.describe()

count    160.000000
mean       7.499019
std        6.055345
min      -10.924000
25%        8.466750
50%        9.388500
75%       10.341000
max       10.963000
Name: diameter, dtype: float64

In [13]:
# Liste des statuts possibles, pour pouvoir les mapper vers 1 ou 0
bronze_diameters.detection_status.value_counts()

detection_status
OK           22
Err          19
yes          19
ERror        17
oK           14
FAILURE      13
Ok           12
ok           12
Good          6
E R R O R     5
Fail          5
good          5
gooD          5
GOod          4
Godo          2
Name: count, dtype: int64

Créer un nouveau dataframe qui contient les données mises en forme et corrigées :
- colonne `timestamp` de type "datetime" Pandas
- élimination des diamètres invalides
- remplacement de la colonne `detection_status` par un entier pouvant prendre les valeurs 1 (état OK) ou 0 (état défaillant)

In [14]:
# Une solution possible pour les statuts : une fonction qui fait le mapping chaîne -> 1 ou 0

def map_detection_status(status: str) -> int:
    status = status.lower()
    if status in ['ok', 'yes']:
        return 1
    elif status.startswith('g'):
        return 1
    elif status.startswith('f') or status.startswith('e'):
        return 0
    else:
        # Si jamais on a oublié un cas, on force la conversion à échouer pour enrichir le mapping
        raise Exception(f'Cannot map "{status}"!')

# 1. Elimination des lignes aberrantes
bronze_diameters_rect = bronze_diameters[bronze_diameters.diameter > 0]

bronze_diameters_rect = bronze_diameters_rect.assign(
    # 2. conversion des timestamps
    timestamp=pd.to_datetime(bronze_diameters_rect.timestamp),
    # 3. conversion des statuts
    detection_status=bronze_diameters_rect.detection_status.apply(map_detection_status)
)

In [15]:
bronze_diameters_rect

,timestamp,variety_id,diameter,detection_status
0,2026-01-10 14:52:03,12,10.305,1
1,2026-01-10 14:53:00,7,8.995,1
2,2026-01-10 14:54:05,12,10.099,0
3,2026-01-10 14:55:16,12,9.115,0
4,2026-01-10 14:56:12,16,9.875,0
...,...,...,...,...
35,2026-01-10 16:47:12,6,10.804,1
36,2026-01-10 16:48:16,1,8.363,0
37,2026-01-10 16:49:15,2,10.880,0
38,2026-01-10 16:50:07,12,10.681,1


In [16]:
# On vérifie que les types sont bon : ok pour `timestamp` et pour `detection_status`
bronze_diameters_rect.dtypes

timestamp           datetime64[ns]
variety_id                   int64
diameter                   float64
detection_status             int64
dtype: object

Sauver ensuite ce nouveau dataframe au format Parquet dans le répertoire `silver/diameters` (pas un fichier, pas de partitionnement, pas d'index).

In [17]:
bronze_diameters_rect.to_parquet(os.path.join(SILVER_DIR, 'diameters', 'diameters.parquet'), index=False)

## Passage vers la couche _Gold_

Recharger séparément les 2 dataframes du répertoire `silver`, et en déduire 2 nouveaux dataframes :
- `gold_ok` qui contient la jointure entre les diamètres et les référentiels
- `gold_error`un qui contient les données de diamètre (uniquement) n'ayant pas pu être jointes aux variétés

Sauver ces 2 fichiers dans le répertoire `gold` (format Parquet, pas de partition). Vous pouvez nommer les fichiers comme les dataframes (ex. `gold_ok.parquet`).

In [18]:
silver_varieties = pd.read_parquet(os.path.join(SILVER_DIR, 'referentials', 'varieties.parquet'))

In [19]:
silver_diameters = pd.read_parquet(os.path.join(SILVER_DIR, 'diameters', 'diameters.parquet'))

In [20]:
gold_ok = pd.merge(silver_diameters, silver_varieties, on=['variety_id'], how='inner')

In [21]:
gold_ok

,timestamp,variety_id,diameter,detection_status,name,texture
0,2026-01-10 14:52:03,12,10.305,1,Reinette grise du Canada,soft
1,2026-01-10 14:53:00,7,8.995,1,Pink Lady,medium
2,2026-01-10 14:54:05,12,10.099,0,Reinette grise du Canada,soft
3,2026-01-10 14:55:16,12,9.115,0,Reinette grise du Canada,soft
4,2026-01-10 14:56:12,16,9.875,0,Belchard,soft
...,...,...,...,...,...,...
123,2026-01-10 16:47:12,6,10.804,1,Elstar,hard
124,2026-01-10 16:48:16,1,8.363,0,Gala,medium
125,2026-01-10 16:49:15,2,10.880,0,Granny Smith,medium
126,2026-01-10 16:50:07,12,10.681,1,Reinette grise du Canada,soft


In [22]:
# Un moyen pour avoir les jointures en échec : faire une jointure externe ("left") et repérer les lignes qui
# ont apporté des valeurs nulles (par ex. dans le champ `name` qui vient de la table de droite, i.e. les variétés)
gold_error = pd.merge(silver_diameters, silver_varieties, on=['variety_id'], how='left')
gold_error = gold_error[gold_error.name.isnull()]

# Ne garde que les colonnes qui viennent de `silver_diameters`
gold_error = gold_error[silver_diameters.columns]

In [23]:
gold_error

,timestamp,variety_id,diameter,detection_status
6,2026-01-10 14:58:15,14,10.099,0
22,2026-01-10 15:16:08,5,10.194,1
29,2026-01-10 15:24:18,5,9.517,1
33,2026-01-10 15:28:11,5,9.051,1
44,2026-01-10 14:21:18,5,8.467,0
53,2026-01-10 14:32:07,14,10.602,0
59,2026-01-10 14:40:12,14,9.708,1
67,2026-01-10 14:48:08,5,10.290,0
68,2026-01-10 14:50:14,5,8.343,1
75,2026-01-10 15:38:19,14,8.460,1


In [24]:
gold_ok.to_parquet(os.path.join(GOLD_DIR, 'gold_ok.parquet'), index=False)
gold_error.to_parquet(os.path.join(GOLD_DIR, 'gold_error.parquet'), index=False)